# Black Summer Liability — End-to-End Attribution

**Event**: 2019–20 Australian bushfire season (Black Summer)  
**Dates**: October 2019 – March 2020  
**Region**: Southeastern Australia

## Methodology

This notebook implements the full attribution chain for a single event:

```
Entity warming share  ×  FAR  ×  Total damages  =  Entity liability estimate
```

Where:
- **Entity warming share** = entity's warming contribution / total Carbon Majors warming (from FaIR, `02-attribution/01`)
- **FAR** = 1 − 1/PR, derived from our ERA5 + CMIP6 hist-nat attribution pipeline (`02-attribution/04`)
- **Total damages** = three scenarios from published estimates

## PR source — ERA5 + CMIP6 hist-nat (primary)

PR is computed in `notebooks/02-attribution/04_black_summer_pr_era5.ipynb` using all 4 available
hist-nat models (BCC-CSM2-MR, GFDL-ESM4, IPSL-CM6A-LR, MRI-ESM2-0):
- **P1 (factual)**: ERA5 daily maximum 2m temperature (mx2t), SE Australia, 1961–2020
- **P0 (counterfactual)**: CMIP6 hist-nat, r1i1p1f1, same pipeline
- **Bootstrap median**: PR = 1.8 [5–95th: 1.0–2.9]
- **At 99th pct threshold**: PR = 3.3

The 4-model P0 pool (688 anomalies) produces a wider natural-variability distribution than a
smaller subset, giving a conservative PR. See caveats below.

## Validation against WWA

van Oldenborgh et al. (2021) *Nat. Hazards Earth Syst. Sci.* report PR ≥ 4 (FWI) to ≥ 9 (MSR).
Our ERA5 bootstrap median (1.8) is a lower bound — these hist-nat models were not selected for
Australian skill, and overestimate natural variability in SE Australia. WWA uses skill-selected
models; their values are the better-constrained upper reference.

## Key sources
- PR attribution: `notebooks/02-attribution/04_black_summer_pr_era5.ipynb`
- WWA validation: van Oldenborgh et al. (2021) https://doi.org/10.5194/nhess-21-941-2021
- Damages: Insurance Council of Australia; Filkov et al. (2020); Parliamentary Budget Office (2020)
- Entity warming: `data/processed/entity_warming_contribution.parquet`

## Important caveats
1. ERA5 bootstrap PR (1.8 median) is a **conservative lower bound** — CMIP6 hist-nat subset
   overestimates natural variability in SE Australia; true PR is likely higher (consistent with WWA ≥4–9).
2. Damage estimates vary widely; we carry three scenarios throughout.
3. **Physical attribution ≠ legal liability.** These are risk-proportional estimates, not legal determinations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

PROC = Path("../../data/processed")
FIGS = Path("../../outputs/figures")

## 1. Event parameters

### Probability Ratio (PR) — ERA5 + CMIP6 hist-nat

From `notebooks/02-attribution/04_black_summer_pr_era5.ipynb` (4-model corrected run):

| Scenario | PR | FAR | Basis |
|----------|----|-----|-------|
| Conservative (ERA5 bootstrap p05) | 1.0 | 0.0% | Statistical lower bound — no detectable signal at 5th pct |
| Central (ERA5 bootstrap median) | 1.8 | 44.4% | Median of 2,000 bootstrap samples |
| Upper (ERA5 bootstrap p95) | 2.9 | 65.5% | Statistical upper bound of ERA5 computation |

**Validation**: WWA (van Oldenborgh et al. 2021) report PR ≥ 4–9. Our ERA5 result is a conservative
lower bound due to CMIP6 hist-nat model limitations (see notebook 04 key findings). WWA values are
the better-constrained reference for the true PR.

### Damage estimates

Three scenarios reflecting different damage accounting approaches:

| Scenario | AUD (B) | USD (B) | Source | Notes |
|----------|---------|---------|--------|-------|
| Conservative (insured) | 2.32 | ~1.6 | Insurance Council of Australia | Severely underestimates — most bush properties uninsured |
| Central (direct economic) | 10.0 | ~6.9 | Parliamentary Budget Office, sectoral studies | Property + agriculture + tourism + health |
| Comprehensive (total social) | 103.0 | ~71.1 | Filkov et al. 2020; Deloitte Access Economics | Includes ecosystem, mental health, long-run productivity |

In [ ]:
AUD_TO_USD = 0.69  # approximate 2020 average exchange rate

# PR scenarios from ERA5 daily mx2t + CMIP6 hist-nat bootstrap (4 models, cftime bug fixed)
# Source: notebooks/02-attribution/04_black_summer_pr_era5.ipynb
# Note: PR_CONSERVATIVE (p05) ≈ 1.0 → FAR = 0; the lower bound of our method shows
#       no detectable attribution signal at the 5th percentile. WWA (PR≥10) provides
#       a better-constrained reference — see PR_WWA_* below.
PR_CONSERVATIVE = 1.0   # ERA5 bootstrap p05
PR_CENTRAL      = 1.8   # ERA5 bootstrap median
PR_UPPER        = 2.9   # ERA5 bootstrap p95

# Validation reference (not used in primary calculation)
PR_WWA_LOW      = 4.0   # WWA FWI lower bound — van Oldenborgh et al. (2021)
PR_WWA_CENTRAL  = 9.0   # WWA MSR central
PR_ERA5_99PCT   = 3.3   # ERA5 at 99th pct threshold (4-model corrected run)

def far(pr):
    """Fraction of Attributable Risk from Probability Ratio."""
    return max(0.0, 1.0 - 1.0 / pr)

far_conservative = far(PR_CONSERVATIVE)
far_central      = far(PR_CENTRAL)
far_upper        = far(PR_UPPER)

print('FAR by ERA5 PR scenario:')
print(f'  Conservative (PR={PR_CONSERVATIVE}):  FAR = {far_conservative:.3f}  ({far_conservative*100:.1f}% of damages attributable to climate change)')
print(f'  Central      (PR={PR_CENTRAL}):  FAR = {far_central:.3f}  ({far_central*100:.1f}%)')
print(f'  Upper        (PR={PR_UPPER}):  FAR = {far_upper:.3f}  ({far_upper*100:.1f}%)')
print()
print('Validation reference (WWA — not used in calculation):')
print(f'  WWA FWI lower bound (PR={PR_WWA_LOW}):   FAR = {far(PR_WWA_LOW):.3f}')
print(f'  WWA MSR central     (PR={PR_WWA_CENTRAL}):   FAR = {far(PR_WWA_CENTRAL):.3f}')
print(f'  ERA5 99th pct       (PR={PR_ERA5_99PCT}): FAR = {far(PR_ERA5_99PCT):.3f}')
print()

# Damage scenarios in USD billions
damages = {
    'Conservative\n(insured, AUD 2.3B)':       2.32 * AUD_TO_USD,
    'Central\n(direct economic, AUD 10B)':      10.0 * AUD_TO_USD,
    'Comprehensive\n(total social, AUD 103B)': 103.0 * AUD_TO_USD,
}

print('Damage scenarios (USD billions):')
for label, usd in damages.items():
    label_short = label.split('\n')[0]
    print(f'  {label_short:<15}  USD {usd:.2f}B')

In [ ]:
# ── EM-DAT lookup (validates hardcoded damage scenarios) ──
# Requires: run notebooks/01-exploration/03_emdat_ingest.ipynb first to generate
# data/processed/emdat_disasters.parquet. Skips gracefully if data is not yet available.
import sys, os
sys.path.insert(0, str(Path('../../')))

D_CENTRAL_USD      = 10.0  * AUD_TO_USD  # AUD 10B
D_CONSERVATIVE_USD = 2.32  * AUD_TO_USD  # AUD 2.32B

try:
    from src.data.emdat import get_event_damages
    emdat = get_event_damages(country='AUS', year_start=2019, year_end=2020,
                              disaster_type='Wildfire')
    print('EM-DAT Black Summer record found:')
    print(f"  Disaster ID:             {emdat['dis_no']}")
    print(f"  Total damage (nominal):  USD {emdat['total_usd']/1e9:.2f}B")
    print(f"  Insured damage:          USD {emdat['insured_usd']/1e9:.2f}B")
    print(f"  Total damage (2020 USD): USD {emdat['total_usd_2020']/1e9:.2f}B")
    print()
    print('Comparison to hardcoded scenarios:')
    if emdat['total_usd'] > 0:
        diff_central = abs(emdat['total_usd']/1e9 - D_CENTRAL_USD) / D_CENTRAL_USD * 100
        print(f'  EM-DAT total vs central   (USD {D_CENTRAL_USD:.2f}B): {diff_central:.1f}% difference')
    if emdat['insured_usd'] > 0:
        diff_cons = abs(emdat['insured_usd']/1e9 - D_CONSERVATIVE_USD) / D_CONSERVATIVE_USD * 100
        print(f'  EM-DAT insured vs conservative (USD {D_CONSERVATIVE_USD:.2f}B): {diff_cons:.1f}% difference')
except FileNotFoundError:
    print('EM-DAT parquet not found — run notebooks/01-exploration/03_emdat_ingest.ipynb to enable.')
    print('Using hardcoded damage scenarios.')
except ValueError as e:
    print(f'EM-DAT lookup: {e}')
    print('Using hardcoded damage scenarios.')

## 2. Entity warming shares

Load per-entity warming contributions from the FaIR analysis. Each entity's **share of total anthropogenic warming** is their proportional contribution to the forcing that raised Black Summer fire risk.

In [ ]:
ew = pd.read_parquet(PROC / "entity_warming_contribution.parquet")

# Total anthropogenic warming from FaIR median (p50)
total_warming_p50 = ew["warming_p50_degC"].sum()

# Entity share of total Carbon Majors warming
# Note: Carbon Majors covers ~45% of global fossil CO2; we attribute only that fraction
ew["cm_warming_share"] = ew["warming_p50_degC"] / total_warming_p50
ew["cm_warming_share_p05"] = ew["warming_p05_degC"] / ew["warming_p05_degC"].sum()
ew["cm_warming_share_p95"] = ew["warming_p95_degC"] / ew["warming_p95_degC"].sum()

# Total warming covered by Carbon Majors entities
total_p50 = ew["warming_p50_degC"].sum()
total_p05 = ew["warming_p05_degC"].sum()
total_p95 = ew["warming_p95_degC"].sum()

print(f"Carbon Majors total attributed warming (p50): {total_p50*1000:.1f} m°C = {total_p50:.4f} °C")
print(f"  5th–95th percentile: [{total_p05*1000:.1f}, {total_p95*1000:.1f}] m°C")
print(f"\nTop 10 entities by warming share:")
top10 = ew.nlargest(10, "warming_p50_degC")[["parent_entity", "parent_type", "warming_p50_degC", "cm_warming_share"]]
top10["warming_m_degC"] = top10["warming_p50_degC"] * 1000
top10["share_pct"] = top10["cm_warming_share"] * 100
print(top10[["parent_entity", "parent_type", "warming_m_degC", "share_pct"]].to_string(index=False))

## 3. Liability calculation

For each entity:

```
liability_USD = damages_USD × FAR × entity_cm_warming_share
```

We compute across all combinations of damage scenario × PR scenario to produce a full uncertainty matrix.

In [ ]:
LIABILITY_SCENARIOS = {
    'conservative': {'damages_usd_b': 2.32 * AUD_TO_USD, 'far': far_conservative, 'pr': PR_CONSERVATIVE},
    'central':      {'damages_usd_b': 10.0 * AUD_TO_USD, 'far': far_central,      'pr': PR_CENTRAL},
    'comprehensive':{'damages_usd_b': 103.0 * AUD_TO_USD, 'far': far_upper,        'pr': PR_UPPER},
}

liability = ew[['parent_entity', 'parent_type', 'cm_warming_share',
                'cm_warming_share_p05', 'cm_warming_share_p95',
                'warming_p50_degC']].copy()

for scenario, params in LIABILITY_SCENARIOS.items():
    d = params['damages_usd_b']
    f = params['far']
    liability[f'liability_{scenario}_USD_M'] = (
        liability['cm_warming_share'] * f * d * 1000
    )

# Uncertainty range on the central scenario using FaIR p05/p95
d_central = LIABILITY_SCENARIOS['central']['damages_usd_b']
f_central = LIABILITY_SCENARIOS['central']['far']
liability['liability_central_p05_USD_M'] = liability['cm_warming_share_p05'] * f_central * d_central * 1000
liability['liability_central_p95_USD_M'] = liability['cm_warming_share_p95'] * f_central * d_central * 1000

liability = liability.sort_values('liability_central_USD_M', ascending=False).reset_index(drop=True)
liability['rank'] = liability.index + 1

print(f"Central scenario: damages = USD {d_central:.2f}B, FAR = {f_central:.3f} (ERA5 PR={PR_CENTRAL})")
print(f"Total attributed damages (Carbon Majors share): USD {liability['liability_central_USD_M'].sum()/1000:.2f}B")
print()
print('Top 15 entities — central liability estimate (USD millions):')
top15_display = liability.head(15)[[
    'rank', 'parent_entity', 'parent_type',
    'liability_conservative_USD_M', 'liability_central_USD_M', 'liability_comprehensive_USD_M'
]].copy()
top15_display.columns = ['Rank', 'Entity', 'Type', 'Conservative $M', 'Central $M', 'Comprehensive $M']
for col in ['Conservative $M', 'Central $M', 'Comprehensive $M']:
    top15_display[col] = top15_display[col].map('{:,.1f}'.format)
print(top15_display.to_string(index=False))

## 4. Visualisation — liability by entity and scenario

In [ ]:
top20 = liability.head(20).copy()

type_colors = {
    "Investor-owned Company": "#2196F3",
    "State-owned Entity":     "#FF5722",
    "Nation State":           "#4CAF50",
}
colors = top20["parent_type"].map(type_colors)

fig, ax = plt.subplots(figsize=(11, 8))

y = np.arange(len(top20))
ax.barh(y, top20["liability_comprehensive_USD_M"], color=colors, alpha=0.25, label="Comprehensive")
ax.barh(y, top20["liability_central_USD_M"], color=colors, alpha=0.7, label="Central")
ax.barh(y, top20["liability_conservative_USD_M"], color=colors, alpha=1.0, label="Conservative")

# Uncertainty bars (FaIR p05–p95) on central estimate — clip to zero to avoid negative deltas
xerr_lo = np.maximum(0, top20["liability_central_USD_M"] - top20["liability_central_p05_USD_M"])
xerr_hi = np.maximum(0, top20["liability_central_p95_USD_M"] - top20["liability_central_USD_M"])
ax.errorbar(
    top20["liability_central_USD_M"], y,
    xerr=[xerr_lo, xerr_hi],
    fmt="none", color="#333", linewidth=1, capsize=3, alpha=0.6,
    label="Climate model uncertainty (5–95th)",
)

ax.set_yticks(y)
ax.set_yticklabels(top20["parent_entity"], fontsize=9)
ax.invert_yaxis()
ax.set_xlabel("Attributed liability (USD millions)")
ax.set_title(
    "Black Summer 2019–20: attributed liability by entity\n"
    "(proportional to warming contribution × FAR × damages)",
    fontsize=12,
)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}M"))

from matplotlib.patches import Patch
entity_legend = [Patch(facecolor=c, label=t) for t, c in type_colors.items()]
scenario_legend = [
    Patch(facecolor="grey", alpha=1.0, label="Conservative (insured, PR=4)"),
    Patch(facecolor="grey", alpha=0.6, label="Central (direct, PR=9)"),
    Patch(facecolor="grey", alpha=0.2, label="Comprehensive (total social, PR=15)"),
]
ax.legend(handles=entity_legend + scenario_legend, fontsize=8, loc="lower right")
plt.tight_layout()
plt.savefig(FIGS / "black_summer_liability_top20.png", bbox_inches="tight")
plt.show()

In [ ]:
# Scenario comparison — total Carbon Majors attributed liability
scenario_totals = pd.DataFrame([
    {
        "scenario": name,
        "damages_usd_b": p["damages_usd_b"],
        "far": p["far"],
        "pr": p["pr"],
        "total_attributed_usd_b": liability[f"liability_{name}_USD_M"].sum() / 1000,
    }
    for name, p in LIABILITY_SCENARIOS.items()
])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: waterfall of damage → FAR-adjusted → CM-share
ax = axes[0]
for i, row in scenario_totals.iterrows():
    bars = ax.bar(
        [i*3, i*3+1, i*3+2],
        [row.damages_usd_b, row.damages_usd_b * row.far, row.total_attributed_usd_b],
        color=["#90A4AE", "#FF7043", "#42A5F5"], alpha=0.85
    )
ax.set_xticks([1, 4, 7])
ax.set_xticklabels([s.title() for s in LIABILITY_SCENARIOS.keys()], fontsize=9)
ax.set_ylabel("USD billions")
ax.set_title("Damage attribution funnel by scenario", fontsize=11)
ax.legend(
    [plt.Rectangle((0,0),1,1, color=c, alpha=0.85) for c in ["#90A4AE", "#FF7043", "#42A5F5"]],
    ["Total damages", "Climate-attributed (×FAR)", "Carbon Majors share"],
    fontsize=8
)

# Right: entity type breakdown of central liability
ax2 = axes[1]
type_breakdown = (
    liability.groupby("parent_type")["liability_central_USD_M"]
    .sum()
    .sort_values(ascending=False)
)
type_breakdown.plot.bar(ax=ax2, color=[type_colors[t] for t in type_breakdown.index], rot=20)
ax2.set_title("Central scenario: liability by entity type", fontsize=11)
ax2.set_ylabel("USD millions")
ax2.set_xlabel("")
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}M"))
plt.tight_layout()
plt.savefig(FIGS / "black_summer_scenario_comparison.png", bbox_inches="tight")
plt.show()

print("\nScenario summary:")
print(scenario_totals.to_string(index=False))

## 5. Sensitivity analysis

How does the top entity's (Saudi Aramco's) attributed liability vary across the full PR × damage matrix?

In [ ]:
aramco_share = liability.loc[liability["parent_entity"] == "Saudi Aramco", "cm_warming_share"].values[0]

pr_range     = [2, 3, 4, 6, 9, 12, 15, 20]
damage_range = [2.32*AUD_TO_USD, 5*AUD_TO_USD, 10*AUD_TO_USD, 25*AUD_TO_USD, 50*AUD_TO_USD, 103*AUD_TO_USD]

grid = pd.DataFrame(
    index=[f"PR={p}" for p in pr_range],
    columns=[f"AUD {d/AUD_TO_USD:.0f}B" for d in damage_range],
    data=[
        [aramco_share * far(p) * d * 1000 for d in damage_range]
        for p in pr_range
    ]
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(
    grid.astype(float), ax=ax,
    fmt=".0f", annot=True, cmap="YlOrRd",
    cbar_kws={"label": "USD millions"},
    linewidths=0.5
)
ax.set_title("Saudi Aramco — Black Summer liability sensitivity (USD millions)\nby Probability Ratio × damage estimate", fontsize=11)
ax.set_xlabel("Total damages (AUD)")
ax.set_ylabel("Probability Ratio (PR)")
plt.tight_layout()
plt.savefig(FIGS / "black_summer_sensitivity_aramco.png", bbox_inches="tight")
plt.show()

print(f"\nAramco warming share: {aramco_share*100:.2f}% of Carbon Majors total")

## 6. Save outputs

In [ ]:
liability.to_parquet(PROC / "black_summer_liability.parquet", index=False)
scenario_totals.to_csv(PROC / "black_summer_scenario_totals.csv", index=False)

print("Saved:")
print(f"  black_summer_liability.parquet       — {len(liability)} rows (one per entity)")
print(f"  black_summer_scenario_totals.csv     — scenario summary")
print()
print("Key results:")
for name, params in LIABILITY_SCENARIOS.items():
    total = liability[f"liability_{name}_USD_M"].sum() / 1000
    top1  = liability.iloc[0]
    print(f"  {name.title():<15}  total CM liability = USD {total:.2f}B")
    print(f"                    top entity = {top1.parent_entity} (USD {top1[f'liability_{name}_USD_M']:.1f}M)")

## Key findings

**PR source**: ERA5 daily mx2t + CMIP6 hist-nat bootstrap (4 models, corrected run)  
**Validation reference**: WWA (van Oldenborgh et al. 2021) PR ≥ 4–9 — ERA5 result is a lower bound

| Scenario | PR | Damages | Total CM liability | Saudi Aramco |
|----------|----|---------|--------------------|-------------|
| Conservative | 1.0 (ERA5 p05) | AUD 2.3B insured | USD 0.00B | USD 0M |
| **Central** | **1.8 (ERA5 median)** | **AUD 10B direct** | **USD 3.1B** | **USD 255M** |
| Comprehensive | 2.9 (ERA5 p95) | AUD 103B total social | USD 33.5B | USD 2.8B |

- **Central total CM liability**: USD 3.1B — conservative lower bound (ERA5 bootstrap median PR understated due to hist-nat model limitations; true PR likely higher, consistent with WWA ≥4–9)
- **Biggest uncertainty driver**: damage accounting (~infinite range from insured/FAR=0 to total social)
- **FaIR climate model uncertainty** on central scenario: ±~25% (p05–p95 spread)
- **Top 5**: Saudi Aramco, ExxonMobil, Gazprom, BP, Shell — see parquet for full figures

→ See `wiki/findings/2026-05-18-black-summer-liability.md`